<a href="https://colab.research.google.com/github/HuchpechTW/public-colabs/blob/main/SPICEUP_HSVZ_Lab_v2_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
from google.colab import auth
from google.auth import default
import gspread
import pandas as pd
import numpy as np

# 这一段主要是拼接item group id
# 然后单个计算item的label
# 再计算整体的group performance

# 1. 读取你云盘里的原始数据 (请根据实际文件名修改路径)
# 假设 Ads 表格里有 'Item ID', 'Cost', 'Conversion Value' 三列
performance_excel='Item-Performance-260226-260327'

item_group_folder = '/content/drive/MyDrive/0-SPICEUP-Huchpech/9-GMC-ItemGroupID/'
# 从gmc 到处的含有item group id 的数据
item_group_excel='gmc_products_2026-03-28.tsv'


# 1. 验证你的账号权限（运行会弹窗让你点授权）
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 2. 读取ads item performance数据
worksheet = gc.open(performance_excel).sheet1
rows = worksheet.get_all_values()
ads_df = pd.DataFrame.from_records(rows[1:], columns=rows[0])
ads_df['Item ID'] = ads_df['Item ID'].astype(str)
print('Google Ads item performancedata')
print(ads_df.head()) # 打印出来看看，瞬间搞定


# 3. 读取gmc item group id数据
# 假设 GMC 表格里有 'id', 'item_group_id' 两列
gmc_df = pd.read_csv(item_group_folder+item_group_excel, sep='\t')
# 测试与处理item group id的空值问题
nan_count = gmc_df['item group id'].isna().sum()
if nan_count > 0:
  print("👉 结论：正是这几个空值，迫使 Pandas 开启了底层保护机制，将整列的整数强行转化为了带小数点的浮点数 (float64)。")
  gmc_df['item group id'] = (
    pd.to_numeric(gmc_df['item group id'], errors='coerce')
    .astype('Int64')
    .astype(str)
    .replace('<NA>', '')
  )

gmc_df['id'] = gmc_df['id'].astype(str)
gmc_df['item group id'] = gmc_df['item group id'].astype(str)

print('GMC item group id data')
print(gmc_df.head()) # 打印出来看看


# 5. 数据缝合 (VLOOKUP 的 Python 替代版)
# 把 GMC 里的父体 ID 拼接到 Ads 的表现数据上
merged_df = pd.merge(ads_df, gmc_df[['id', 'item group id']],
                     left_on='Item ID', right_on='id', how='left')

merged_df['Clicks'] = pd.to_numeric(merged_df['Clicks'].astype(str).str.replace(r'[^\d.]', '', regex=True), errors='coerce')
merged_df['Conversions'] = pd.to_numeric(merged_df['Conversions'].astype(str).str.replace(r'[^\d.]', '', regex=True), errors='coerce')
merged_df['Impr.'] = pd.to_numeric(merged_df['Impr.'].astype(str).str.replace(r'[^\d.]', '', regex=True), errors='coerce')
merged_df['Cost'] = pd.to_numeric(merged_df['Cost'].astype(str).str.replace(r'[^\d.]', '', regex=True), errors='coerce')
merged_df['Conv. value'] = pd.to_numeric(merged_df['Conv. value'].astype(str).str.replace(r'[^\d.]', '', regex=True), errors='coerce')
# print('Merged data')
print(merged_df.head())


# 6. 按手机壳父体 (Item Group ID) 聚合总花费和总产出
merged_df['item group id'] = merged_df['item group id'].replace('', np.nan)
merged_df['item group id'] = merged_df['item group id'].fillna('Unmatched')
parent_df = merged_df.groupby('item group id').agg(
    total_cost=('Cost', 'sum'),
    total_value=('Conv. value', 'sum'),
    total_clicks=('Clicks', 'sum'),
    total_conv=('Conversions', 'sum'),
    total_impr=('Impr.', 'sum')
).reset_index()
# print(parent_df.head())

import numpy as np
assert np.isclose(merged_df['Cost'].sum(), parent_df['total_cost'].sum(), rtol=1e-10), 'Aggregation后总消耗要相同'

# assert merged_df['Cost'].sum() == parent_df['total_cost'].sum(), 'Aggregation后总消耗要相同'

# 7. 计算Aggerate之后的真实 ROAS
# 避免除以 0 的报错
parent_df['roas'] = parent_df['total_value'] / parent_df['total_cost'].replace(0, 1)


Google Ads item performancedata
                                               Image  \
0  http://t3.gstatic.com/shopping?q=tbn:ANd9GcQyW...   
1  http://t1.gstatic.com/shopping?q=tbn:ANd9GcSNA...   
2  http://t1.gstatic.com/shopping?q=tbn:ANd9GcSPP...   
3  http://t2.gstatic.com/shopping?q=tbn:ANd9GcTge...   
4  http://t0.gstatic.com/shopping?q=tbn:ANd9GcQH_...   

                                               Title         Item ID Clicks  \
0  Minimalist Magnetic Phone Case – Soft Touch Li...  48160872857815      0   
1     Limited Phone Case | ST Label 1, iPhone 12 Pro  47326360338647      0   
2  SPICEUP STUDIO | Limited Phone Case | Ricky & ...  45956279042263      0   
3  SPICEUP STUDIO | Leather Phone Case | KE Purpl...  44169854550231      0   
4  Designer APE Green Leather - Hypebeast Phone C...  44170081403095      8   

  Impr.    CTR Currency code Avg. CPC  Cost Conversions Conv. value  \
0    43     0%           USD        0     0           0           0   
1    86     0%

In [14]:
targetROAS = 1.5
targetCPA = 15
minCostThreshold = 2 * targetCPA
nearhero_lower_bound = 1.30
villains_bound = 1.14

HERO_LABEL='Heroes';
CONTENDER_LABEL='Contenders';
SIDEKICK_LABEL='Sidekicks';
EXPLORATION_LABEL='Explorers';
VILLAINS_LABEL='Villains';



def assign_group(row):
    cost = row['total_cost']
    roas = row['roas']
    conv = row['total_conv']

    maturity = cost / minCostThreshold

    if cost < minCostThreshold:
        return EXPLORATION_LABEL
    elif roas >= targetROAS:
        return HERO_LABEL
    elif roas >= nearhero_lower_bound:
        return CONTENDER_LABEL
    elif roas < villains_bound:
        return VILLAINS_LABEL
    else:
        return SIDEKICK_LABEL

parent_df['group_label'] = parent_df.apply(assign_group, axis=1)
print(parent_df.head())


def assign_item(row):
    cost = row['Cost']
    roas = float(row['Conv. value / cost'])
    conv = row['Conversions']

    if cost < minCostThreshold:
        return EXPLORATION_LABEL
    elif roas >= targetROAS:
        return HERO_LABEL
    elif roas >= nearhero_lower_bound:
        return CONTENDER_LABEL
    elif roas < villains_bound:
        return VILLAINS_LABEL
    else:
        return SIDEKICK_LABEL

merged_df['sku_label']=merged_df.apply(assign_item, axis=1)
merged_df = merged_df.merge(parent_df[['item group id', 'group_label']], on='item group id', how='left')


def update_ads_label(row):
    productLevel = row['group_label'];
    skuLevel = row['sku_label'];

    if productLevel==HERO_LABEL or productLevel==CONTENDER_LABEL:
        return skuLevel;

    if productLevel==SIDEKICK_LABEL or productLevel==EXPLORATION_LABEL:
        if skuLevel==HERO_LABEL:
            return CONTENDER_LABEL
        else:
            return skuLevel

    if productLevel==VILLAINS_LABEL:
        if skuLevel==HERO_LABEL:
            return EXPLORATION_LABEL
        else:
            return VILLAINS_LABEL

merged_df['custom_label_1'] = merged_df.apply(update_ads_label, axis=1)



print(merged_df.head())



   item group id  total_cost  total_value  total_clicks  total_conv  \
0  6946944778420       12.86          0.0            40         0.0   
1  7447147217111        0.45          0.0             1         0.0   
2  7449111920855        3.04          0.0             7         0.0   
3  7449137709271        4.97          0.0            15         0.0   
4  7449138921687        0.07          0.0             0         0.0   

   total_impr  roas group_label  
0        4577   0.0   Explorers  
1         222   0.0   Explorers  
2        1173   0.0   Explorers  
3        1836   0.0   Explorers  
4         287   0.0   Explorers  
                                               Image  \
0  http://t3.gstatic.com/shopping?q=tbn:ANd9GcQyW...   
1  http://t1.gstatic.com/shopping?q=tbn:ANd9GcSNA...   
2  http://t1.gstatic.com/shopping?q=tbn:ANd9GcSPP...   
3  http://t2.gstatic.com/shopping?q=tbn:ANd9GcTge...   
4  http://t0.gstatic.com/shopping?q=tbn:ANd9GcQH_...   

                                

In [22]:
from google.colab import auth
import gspread
from google.auth import default
from datetime import datetime, timezone, timedelta
from googleapiclient.discovery import build

drive_service = build('drive', 'v3', credentials=creds)

FOLDER_ID='1Gki73ptH95N_SdjdXyGiQ_GKdzbHmMCa'

beijing_tz = timezone(timedelta(hours=8))
time_str = datetime.now(beijing_tz).strftime("%Y%m%d%H%M")
# 在云盘中创建一张全新的 Google Sheet
# sh = gc.create('HSZV_Final_Feed')
sh = gc.create('ItemPerformance-Merged-'+time_str)
worksheet = sh.sheet1

# 把刚才处理好的 final_feed 数据转化为列表格式并写入表格
# [final_feed.columns.values.tolist()] 是写入表头
# final_feed.values.tolist() 是写入所有行数据
worksheet.update([merged_df.columns.values.tolist()] + merged_df.fillna('').values.tolist())
print(f"Merged数据已成功生成: https://docs.google.com/spreadsheets/d/{sh.id}")

file = drive_service.files().get(fileId=sh.id, fields='parents').execute()
previous_parents = ",".join(file.get('parents'))

# 执行移动操作：添加新父目录，移除旧父目录
drive_service.files().update(
    fileId=sh.id,
    addParents=FOLDER_ID,
    removeParents=previous_parents,
    fields='id, parents'
).execute()


sh = gc.create('GroupPerformance-Stats-'+time_str)
worksheet = sh.sheet1
worksheet.update([parent_df.columns.values.tolist()] + parent_df.fillna('').values.tolist())
print(f"Aggregation数据已成功生成: https://docs.google.com/spreadsheets/d/{sh.id}")

file = drive_service.files().get(fileId=sh.id, fields='parents').execute()
previous_parents = ",".join(file.get('parents'))

# 执行移动操作：添加新父目录，移除旧父目录
drive_service.files().update(
    fileId=sh.id,
    addParents=FOLDER_ID,
    removeParents=previous_parents,
    fields='id, parents'
).execute()


Merged数据已成功生成: https://docs.google.com/spreadsheets/d/1DsIy_5XYX8_x3U9TnvB6DHJcPoFwbxWxewdlZO1I2Hc
Aggregation数据已成功生成: https://docs.google.com/spreadsheets/d/1OKWG-fadJY_aIKZX3D31nQW73EvUNCKgcjQkXcmI8RE


{'id': '1OKWG-fadJY_aIKZX3D31nQW73EvUNCKgcjQkXcmI8RE',
 'parents': ['1Gki73ptH95N_SdjdXyGiQ_GKdzbHmMCa']}

In [23]:
from google.colab import auth
import gspread
from google.auth import default

# 9. 整理出最后使用的
# 只保留 GMC 补充馈单需要的两列，并去重
final_feed = merged_df[['id', 'item group id', 'custom_label_1', 'group_label','sku_label']].drop_duplicates()

# 打印前 5 行让你核对一下结果是否精准
print(final_feed.head())

# 在云盘中创建一张全新的 Google Sheet
# sh = gc.create('HSZV_Final_Feed')
# sh = gc.create('HSVZ_Feed_stats-'+time_str)
sh = gc.open('SPICEUP-GMC-Feed-HSVZ')
worksheet = sh.sheet1
worksheet.clear()

worksheet.update([final_feed.columns.values.tolist()] + final_feed.fillna('').values.tolist())

print(f"数据已成功生成！表格链接: https://docs.google.com/spreadsheets/d/{sh.id}")

               id  item group id custom_label_1 group_label  sku_label
0  48160872857815  9148265103575       Villains    Villains  Explorers
1  47326360338647  9076887453911      Explorers   Explorers  Explorers
2  45956279042263  8795525021911      Explorers   Explorers  Explorers
3  44169854550231  8177307091159      Explorers   Explorers  Explorers
4  44170081403095  8177376428247      Explorers      Heroes  Explorers
数据已成功生成！表格链接: https://docs.google.com/spreadsheets/d/1SybzEEAeNeUBT8nWaDwLU4KJS-_CqMq1Kw_5U5RNY5c
